In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

# ---------------------------------------------------------
# 1. Setup & Data Loading (CIFAR-10)
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# CIFAR-10 images are 32x32, but ResNet expects 224x224.
# We resize them so you can use this exact same dataloader for MAE and I-JEPA later.
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet normalization
])

print("Downloading/Loading CIFAR-10...")
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# ---------------------------------------------------------
# 2. Load Backbone & Setup Linear Probe
# ---------------------------------------------------------
# Load a pre-trained ResNet18
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# FREEZE THE BACKBONE (Crucial for Protocol A)
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification layer.
# In PyTorch, new layers have requires_grad=True by default.
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10) # 10 classes for CIFAR-10

model = model.to(device)

# ---------------------------------------------------------
# 3. Training Loop (Only training the final linear layer)
# ---------------------------------------------------------
criterion = nn.CrossEntropyLoss()
# Notice we are ONLY passing model.fc.parameters() to the optimizer!
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

epochs = 3 # Keep it low just to test if the pipeline works

print("\nStarting Linear Probe Training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Print progress every 100 batches
        if (i + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Batch [{i+1}/{len(trainloader)}], Loss: {loss.item():.4f}, Acc: {100 * correct / total:.2f}%")

print("Finished Training Linear Probe!")

# ---------------------------------------------------------
# 4. Clean Up (To prevent OOM errors for your next model)
# ---------------------------------------------------------
# del model
# torch.cuda.empty_cache()

Using device: cuda
Downloading/Loading CIFAR-10...


 40%|███▉      | 67.8M/170M [13:07<26:33, 64.5kB/s]